# 🎂 Party Planning

This project demonstrates solving a Real-World Planning problem using the example of organizing a birthday party, a problem that involves managing multiple tasks simultaneously under various constraints.

### 📝 Problem Details

- **Goal:** Successfully organize a party, which requires preparing food, decorating the venue, and sending invitations to guests before the event starts.
- **Tasks:**
    1.  **Food Preparation:** Must `BuyIngredients` first, then `CookFood`.
    2.  **Decoration:** Must `BuyDecorations` first, then `DecorateRoom`.
    3.  **Guest Invitation:** `SendInvites`.
- **Resources:**
    - **Money:** Used for purchasing items (a consumable resource).
    - **People:** Required for various tasks like cooking and decorating (a reusable resource).
- **Constraints:**
    - Some tasks have a specific order (e.g., buying ingredients before cooking).
    - Resources are limited.

We will model this problem using `RealWorldPlanningProblem` and `HLA` (High-Level Action) from the `aima-python` library. Each `HLA` will have preconditions, effects, duration, and required resources (consume/use).

In [ ]:
# Import necessary libraries from aima-python
# Ensure that the aima/planning.py file is in the correct path.
import sys
import os

# Add the path to the aima library.
# If this notebook is in the same directory as the aima folder, use '.'.
# If it's in a subfolder like 'projects', use '..'.
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from aima.planning import RealWorldPlanningProblem, HLA
from aima.utils import expr


def party_planning_challenge():
    """
    🏆 Challenge: Party Planning Problem
    Models the problem of planning a party with resource and time constraints.
    """
    # --- Define Initial Resources ---
    # Total time available: 8 hours = 480 minutes
    resources = {'Money': 500, 'Time': 480, 'Cooks': 1,
                 'Decorators': 1, 'Planners': 1}

    # --- กำหนด High-Level Actions (HLAs) ---

    # 1. Guest Invitation Task
    send_invites = HLA('SendInvites',
                       precond='At(Home)',
                       effect='InvitesSent',
                       duration=30,
                       use={'Planners': 1})

    # 2. Food Preparation Task Group (Cooking)
    buy_ingredients = HLA('BuyIngredients',
                          precond='At(Home)',
                          effect='Have(Ingredients)',
                          duration=60,
                          consume={'Money': 40},
                          use={'Planners': 1})

    cook_food = HLA('CookFood',
                    precond='Have(Ingredients) & At(Home)',
                    effect='Have(Food)',
                    duration=120,
                    use={'Cooks': 1})

    # 3. Decoration Task Group
    buy_decorations = HLA('BuyDecorations',
                          precond='At(Home)',
                          effect='Have(Decorations)',
                          duration=90,
                          consume={'Money': 100},
                          use={'Planners': 1})

    decorate_room = HLA('DecorateRoom',
                        precond='Have(Decorations) & At(Home)',
                        effect='IsDecorated',
                        duration=120,
                        use={'Decorators': 1})

    # 4. Other Tasks
    buy_drinks = HLA('BuyDrinks',
                     precond='At(Home)',
                     effect='Have(Drinks)',
                     duration=30,
                     consume={'Money': 60})

    setup_music = HLA('SetupMusic',
                      precond='At(Home)',
                      effect='MusicReady',
                      duration=30)

    # Combine all actions
    actions = [send_invites, buy_ingredients, cook_food,
               buy_decorations, decorate_room, buy_drinks, setup_music]

    # Define the sequence of jobs
    food_prep_job = [buy_ingredients, cook_food]
    decor_job = [buy_decorations, decorate_room]
    # Other tasks that have no mandatory order can be done anytime
    other_tasks = [send_invites, buy_drinks, setup_music]

    # --- Create the Planning Problem ---
    initial_state = expr('At(Home) & ~Have(Ingredients) & ~Have(Food) & '
                         '~Have(Decorations) & ~IsDecorated & ~InvitesSent & '
                         '~Have(Drinks) & ~MusicReady')
    goal_state = expr('Have(Food) & IsDecorated & InvitesSent & '
                      'Have(Drinks) & MusicReady')

    return RealWorldPlanningProblem(
        initial=initial_state,
        goals=goal_state,
        actions=actions,
        jobs=[food_prep_job, decor_job, other_tasks],
        resources=resources)


# --- Test the execution ---

# 1. Create a problem instance
party_problem = party_planning_challenge()
# Store initial money to calculate total spent
initial_money = party_problem.resources['Money']

print("🎉 Starting the party planning!")
print(f"Initial state: {party_problem.initial}")
print(f"Goals: {party_problem.goals}")
print(f"Initial resources: {party_problem.resources}")
print("-" * 30)

# 2. Define the execution plan (sequence of actions)
# This is one possible and economical plan.
# In a real scenario, a planner algorithm would find this.
solution_plan = [
    party_problem.actions[0],  # SendInvites (30 min)
    party_problem.actions[1],  # BuyIngredients (60 min, $40)
    party_problem.actions[3],  # BuyDecorations (90 min, $100)
    party_problem.actions[5],  # BuyDrinks (30 min, $60)
    party_problem.actions[6],  # SetupMusic (30 min)
    party_problem.actions[2],  # CookFood (120 min)
    party_problem.actions[4]   # DecorateRoom (120 min)
]

print("🚀 Executing the plan...")
total_time_used = 0

try:
    for action in solution_plan:
        # Simulate time usage
        total_time_used += action.duration
        time_left = party_problem.resources['Time'] - (total_time_used - action.duration)
        if total_time_used > party_problem.resources['Time']:
            raise Exception(
                f"Not enough time for '{action.name}'! "
                f"Needs {action.duration} min, but only {time_left} min left."
            )

        party_problem.act(action)
        print(f"  ✅  Successfully completed '{action.name}'!")
        if 'Money' in action.consumes:
            print(f"      - Money remaining: ${party_problem.resources['Money']}")
        print(f"      - Total time used: {total_time_used} minutes")

    print("-" * 30)

    # 3. Check if the goal is reached
    if party_problem.goal_test():
        print("✅ Success! The party is ready!")
    else:
        print("❌ Not finished! Something is missing.")

    print(f"Final state: {party_problem.initial}")
    money_spent = initial_money - party_problem.resources['Money']
    print(f"Total money spent: ${money_spent}")
    print(f"Total time spent: {total_time_used} minutes (out of 480 minutes)")

except Exception as e:
    print(f"An error occurred: {e}")